# 📱 Galaxy S25 FE - Instagram Reels Glitch Fixer
**How to use on your Phone or PC Browser:**
1. Download the latest stock **Instagram `.apkm` (`arm64-v8a`)** from [APKMirror](https://www.apkmirror.com/apk/instagram/instagram-instagram/).
2. Tap the **Play button (▶️)** below.
3. **Upload Box 1:** Tap **Choose Files** and select your downloaded Instagram `.apkm` file.
4. **Upload Box 2 (Optional - For Updates Only):**
   * **First-time users:** Simply tap **Cancel upload**! The script will automatically generate a brand-new `my-instagram.jks` key for you.
   * **Updating from an older version:** Tap **Choose Files** and select your saved `my-instagram.jks` file so your new APK installs right over your existing app without logging out.

In [ ]:
# @title ▶️ Tap the Play Button Here to Upload & Patch Instagram
import os, sys, glob, struct, zlib, hashlib, zipfile, subprocess, urllib.request
from google.colab import files

APKEDITOR_URL = 'https://github.com/REAndroid/APKEditor/releases/download/V1.4.9/APKEditor-1.4.9.jar'
UBERSIGNER_URL = 'https://github.com/patrickfav/uber-apk-signer/releases/download/v1.3.0/uber-apk-signer-1.3.0.jar'
APKEDITOR_JAR = 'APKEditor.jar'
UBERSIGNER_JAR = 'uber-apk-signer.jar'
KEYSTORE_FILE = 'my-instagram.jks'

print('===================================================================')
print('📦 STEP 1 of 2: Upload your stock Instagram .apkm (or .apk) file')
print('===================================================================')
uploaded_apk = files.upload()
apkm_files = [f for f in uploaded_apk.keys() if f.lower().endswith(('.apkm', '.apk'))]
if not apkm_files:
    raise SystemExit('❌ Error: No .apkm or .apk file was selected! Please tap Play again.')
input_file = apkm_files[0]

print('\n===================================================================')
print('🔑 STEP 2 of 2 (OPTIONAL): Upload your saved my-instagram.jks key')
print('👉 FIRST-TIME USERS: Just tap "Cancel upload" below!')
print('👉 UPDATING USERS: Tap "Choose Files" and select my-instagram.jks')
print('===================================================================')
uploaded_key = files.upload()
jks_files = [f for f in uploaded_key.keys() if f.lower().endswith(('.jks', '.keystore'))]
if jks_files and jks_files[0] != KEYSTORE_FILE:
    os.replace(jks_files[0], KEYSTORE_FILE)

print(f'\n[*] Processing package: {input_file}')
if not os.path.exists(APKEDITOR_JAR):
    print('[*] Downloading APKEditor...')
    urllib.request.urlretrieve(APKEDITOR_URL, APKEDITOR_JAR)
if not os.path.exists(UBERSIGNER_JAR):
    print('[*] Downloading uber-apk-signer...')
    urllib.request.urlretrieve(UBERSIGNER_URL, UBERSIGNER_JAR)

if not os.path.exists(KEYSTORE_FILE):
    print('[*] No existing key uploaded. Generating a brand-new personal key (my-instagram.jks)...')
    subprocess.run([
        'keytool', '-genkeypair', '-v', '-keystore', KEYSTORE_FILE,
        '-alias', 'igkey', '-keyalg', 'RSA', '-keysize', '2048', '-validity', '10000',
        '-storepass', 'instagram123', '-keypass', 'instagram123',
        '-dname', 'CN=User, OU=Mobile, O=Personal, L=City, S=State, C=US'
    ], check=True)
else:
    print('[*] Using your uploaded personal signing key (my-instagram.jks)!')

working_apk = 'merged_temp.apk'
if input_file.lower().endswith('.apkm'):
    print('[*] Merging .apkm bundle into standalone APK...')
    subprocess.run(['java', '-jar', APKEDITOR_JAR, 'm', '-i', input_file, '-o', working_apk], check=True)
else:
    working_apk = input_file

def patch_dex(data: bytearray, name: str) -> bytes:
    modified = False
    for old, new in [(b'video/av01', b'video/av00'), (b'c2.exynos.', b'c2.android')]:
        if old in data:
            cnt = data.count(old)
            data = bytearray(bytes(data).replace(old, new))
            print(f'    [{name}] Patched {cnt}x: {old.decode()} -> {new.decode()}')
            modified = True
    if modified:
        data[12:32] = hashlib.sha1(data[32:]).digest()
        checksum = zlib.adler32(data[12:]) & 0xFFFFFFFF
        data[8:12] = struct.pack('<I', checksum)
    return bytes(data)

unaligned_apk = 'patched_unaligned.apk'
print('[*] Applying 10-byte Exynos AV1 decoder fix to DEX files...')
with zipfile.ZipFile(working_apk, 'r') as zin, zipfile.ZipFile(unaligned_apk, 'w') as zout:
    for item in zin.infolist():
        raw = zin.read(item.filename)
        if item.filename.startswith('classes') and item.filename.endswith('.dex'):
            raw = patch_dex(bytearray(raw), item.filename)
        zout.writestr(item, raw, compress_type=item.compress_type)

print('[*] Zipaligning & signing APK...')
subprocess.run([
    'java', '-jar', UBERSIGNER_JAR, '-a', unaligned_apk,
    '--ks', KEYSTORE_FILE, '--ksAlias', 'igkey',
    '--ksPass', 'instagram123', '--ksKeyPass', 'instagram123',
    '--allowResign', '--out', '.'
], check=True)

final_apk = 'instagram-s25fe-fixed.apk'
signed_out = 'patched_unaligned-aligned-signed.apk'
if os.path.exists(signed_out):
    os.replace(signed_out, final_apk)

print('\n✅ SUCCESS! Downloading your patched APK and personal key to your device...')
files.download(final_apk)
if not jks_files:
    files.download(KEYSTORE_FILE)